In [4]:
import pandas as pd

cases_file = 'year_wise_district_malaria.csv'
vulnerability_file = 'engineered_vulnerability_data.csv'
output_file = 'final_time_series_model_data.csv'

# --- Load Datasets ---
try:
    # 1. Load Time-Series Data
    df_cases = pd.read_csv(cases_file)
    # 2. Load Static Data (Vulnerability Index)
    df_vulnerability = pd.read_csv(vulnerability_file)
except FileNotFoundError as e:
    print(f"Error: File not found. Make sure {e.filename} is in the directory.")
    raise

# --- Merge Dataframes ---
df_final_merged = pd.merge(
    df_cases,
    df_vulnerability,
    on='District',
    how='left'
)
# --- Saving and Inspection ---
df_final_merged.to_csv(output_file, index=False)

print("Successfully merged time-series cases with static vulnerability data.")
print(f"Data saved to: '{output_file}'")
print(f"Final dataset rows: {len(df_final_merged)}")
print("\n--- Final Merged Data Head ---")
print(df_final_merged.head())

Successfully merged time-series cases with static vulnerability data.
Data saved to: 'final_time_series_model_data.csv'
Final dataset rows: 17567

--- Final Merged Data Head ---
   Year   District  Total cases  Total deaths  District code  \
0  2000    Andaman        530.0           0.0            NaN   
1  2000   Nicobars        472.0           1.0            NaN   
2  2000   Adilabad       1798.0           2.0          532.0   
3  2000  Anantapur       1301.0           0.0            NaN   
4  2000   Chittoor        291.0           1.0          554.0   

   No:of villages inhabited  No:of villages uninhabited  Number of towns  \
0                       NaN                         NaN              NaN   
1                       NaN                         NaN              NaN   
2                    1590.0                       135.0             22.0   
3                       NaN                         NaN              NaN   
4                    1455.0                        38.0  

In [6]:
import pandas as pd

df_cleaned = df_final_merged.dropna(how='any')
df_cleaned.to_csv("cleaned_file_of_final_time_series_model_data.csv", index=False)

print("Original shape:", df_final_merged.shape)
print("Cleaned shape :", df_cleaned.shape)


Original shape: (17567, 27)
Cleaned shape : (9612, 27)


In [8]:
# --- 1. Load the Merged Data ---
input_file = 'cleaned_file_of_final_time_series_model_data.csv'
output_file = 'final_model_training_data.csv'

try:
    df = pd.read_csv(input_file)
    print(f"Successfully loaded '{input_file}'.")
except FileNotFoundError:
    print(f"ERROR: File not found. Make sure '{input_file}' is in the directory.")
    raise

# --- 2. Sort Data for Time-Series ---
df = df.sort_values(by=['District', 'Year'], ascending=True)

print("Data sorted by District and Year.")

# --- 3. Create Lagged Features ---
# We will create lags for 1 and 2 years.
lag_features = ['Total cases', 'Total deaths']
lag_periods = [1, 2]            

for col in lag_features:
    for lag in lag_periods:
        new_col_name = f"{col}_Lag_{lag}Y"
        # .shift(lag) moves data down by 'lag' rows, grouped by District
        df[new_col_name] = df.groupby('District')[col].shift(lag)

print("Lagged features (Cases_Lag_1Y, Deaths_Lag_1Y, etc.) created.")

# --- 4. Clean Up NaN Rows ---.
print(f"Original row count: {len(df)}")
df_final = df.dropna()
print(f"New row count after dropping NaN rows: {len(df_final)}")

# --- 5. Save the Final Training Data ---
df_final.to_csv(output_file, index=False)

print(f"\nSuccess! Final model training data saved to: {output_file}")
print("This dataset is now 100% ready for model training.")
print("\n--- Final Data Head with Lagged Features ---")
print(df_final.head())

Successfully loaded 'cleaned_file_of_final_time_series_model_data.csv'.
Data sorted by District and Year.
Lagged features (Cases_Lag_1Y, Deaths_Lag_1Y, etc.) created.
Original row count: 9612
New row count after dropping NaN rows: 8860

Success! Final model training data saved to: final_model_training_data.csv
This dataset is now 100% ready for model training.

--- Final Data Head with Lagged Features ---
      Year  District  Total cases  Total deaths  District code  \
714   2002  Adilabad       1360.0           0.0          532.0   
1080  2003  Adilabad        900.0           0.0          532.0   
1448  2004  Adilabad        758.0           0.0          532.0   
1822  2005  Adilabad        845.0           0.0          532.0   
2198  2006  Adilabad        902.0           0.0          532.0   

      No:of villages inhabited  No:of villages uninhabited  Number of towns  \
714                     1590.0                       135.0             22.0   
1080                    1590.0      